In [1]:
import numpy as np

In [2]:
import pandas as pd

# Replace with the actual path to your file
df = pd.read_csv("all_companies_balance_statements.csv")
df2 = pd.read_csv("all_companies_cashflow_statements.csv")
df3 = pd.read_csv("all_companies_income_statements.csv")
df4 = pd.read_excel("listof90year.xlsx")


In [3]:
# First, rename columns in each DataFrame to avoid collisions (except the join keys)
df_bs = df.add_prefix('bs_')
df_cf = df2.add_prefix('cf_')
df_is = df3.add_prefix('is_')

# But restore the join keys to their original names
for col in ['symbol', 'calendarYear', 'period']:
    df_bs[col] = df[col]
    df_cf[col] = df2[col]
    df_is[col] = df3[col]

# Now merge them one by one on 'symbol', 'calendarYear', and 'period'
combined_df = df_bs.merge(df_cf, on=['symbol', 'calendarYear', 'period'], how='outer')
combined_df = combined_df.merge(df_is, on=['symbol', 'calendarYear', 'period'], how='outer')

In [4]:
combined_df

,bs_Symbol,bs_Unnamed: 1,bs_date,bs_symbol,bs_reportedCurrency,bs_cik,bs_fillingDate,bs_acceptedDate,bs_calendarYear,bs_period,...,is_incomeBeforeTaxRatio,is_incomeTaxExpense,is_netIncome,is_netIncomeRatio,is_eps,is_epsdiluted,is_weightedAverageShsOut,is_weightedAverageShsOutDil,is_link,is_finalLink
0,A,26.0,1998-10-31,A,USD,1090872.0,1998-10-31,1998-10-31 00:00:00,1998.0,FY,...,0.049799,139000000.0,2.570000e+08,0.032319,0.58,0.56,443103448.0,458928571.0,NaN,NaN
1,A,25.0,1999-10-31,A,USD,1090872.0,2000-01-25,2000-01-25 00:00:00,1999.0,FY,...,0.094466,275000000.0,5.120000e+08,0.061457,1.35,1.35,380000000.0,457500000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
2,A,24.0,2000-10-31,A,USD,1090872.0,2001-01-17,2001-01-17 00:00:00,2000.0,FY,...,0.108048,407000000.0,7.570000e+08,0.070268,1.68,1.66,449000000.0,455000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
3,A,23.0,2001-10-31,A,USD,1090872.0,2002-01-22,2002-01-22 00:00:00,2001.0,FY,...,-0.056813,-71000000.0,1.680000e+08,0.020010,0.38,0.38,458000000.0,458000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
4,A,22.0,2002-10-31,A,USD,1090872.0,2002-12-20,2002-12-20 17:27:53,2002.0,FY,...,-0.257404,-525000000.0,-1.032000e+09,-0.171714,-2.22,-2.22,465000000.0,465000000.0,https://www.sec.gov/Archives/edgar/data/109087...,https://www.sec.gov/Archives/edgar/data/109087...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153345,NaN,4.0,2019-12-31,NaN,CNY,1872302.0,2019-12-31,2019-12-30 19:00:00,2019.0,FY,...,0.000000,1587.0,-1.103114e+07,0.000000,-0.28,-0.28,39624617.0,39624500.0,NaN,NaN
153346,NaN,3.0,2020-12-31,NaN,CNY,1872302.0,2020-12-31,2020-12-31 00:00:00,2020.0,FY,...,-17.728034,2293.0,-3.770408e+07,-17.729112,-0.73,-0.73,51895000.0,51951290.0,NaN,NaN
153347,NaN,2.0,2021-12-31,NaN,CNY,1872302.0,2021-12-31,2021-12-31 00:00:00,2021.0,FY,...,-4.435611,1365723.0,-1.763102e+08,-4.470238,-3.40,-3.39,51895000.0,51951290.0,NaN,NaN
153348,NaN,1.0,2022-12-31,NaN,CNY,1872302.0,2023-04-18,2023-04-18 16:31:28,2022.0,FY,...,0.031652,-16847750.0,4.796664e+07,0.048788,0.90,0.89,53244498.0,53665000.0,https://www.sec.gov/Archives/edgar/data/187230...,https://www.sec.gov/Archives/edgar/data/187230...


In [5]:
# Gross Profit / Total Assets
combined_df['gross_profit'] = combined_df['is_revenue'] - combined_df['is_costOfRevenue']
combined_df['gross_ROA'] = combined_df['gross_profit'] / combined_df['bs_totalCurrentAssets']

# EBIT / Market Price (using total stockholders equity as market price proxy)
combined_df['EBIT_to_MP'] = combined_df['is_ebitda'] / combined_df['bs_totalStockholdersEquity']

In [6]:
combined_df['gross_ROA'] = combined_df['gross_ROA'].replace([np.inf, -np.inf], np.nan)
combined_df['EBIT_to_MP'] = combined_df['EBIT_to_MP'].replace([np.inf, -np.inf], np.nan)

In [7]:
# Prepare a list to collect results for each year
ranked_results = []

# Loop over each row in df4 (each year)
for idx, row in df4.iterrows():
    year = row['year']
    prev_year = year - 1

    # Get the tickers for this year, drop NaNs and flatten
    tickers = row[1:].dropna().unique().tolist()

    # Filter combined_df for previous year's financials and relevant tickers
    df_year = combined_df[
        (combined_df['calendarYear'] == prev_year) &
        (combined_df['symbol'].isin(tickers))
    ].copy()

    # Only proceed if there's data
    if df_year.empty:
        continue

    # Rank by gross_ROA (higher is better)
    df_year['rank_gross_ROA'] = df_year['gross_ROA'].rank(ascending=False, method='min')

    # Rank by EBIT_to_MP (higher is better)
    df_year['rank_EBIT_to_MP'] = df_year['EBIT_to_MP'].rank(ascending=False, method='min')

    # Sum ranks
    df_year['combined_score'] = df_year['rank_gross_ROA'] + df_year['rank_EBIT_to_MP']

    # Final ranking based on combined score (lower is better)
    df_year['final_rank'] = df_year['combined_score'].rank(method='min')

    # Add current year for context
    df_year['ranking_year'] = year

    # Save result
    ranked_results.append(df_year)

# Concatenate all results into a single DataFrame
ranked_df = pd.concat(ranked_results, ignore_index=True)

# Optional: sort by year and final rank
ranked_df = ranked_df.sort_values(by=['ranking_year', 'final_rank'])

In [8]:
ranked_df

,bs_Symbol,bs_Unnamed: 1,bs_date,bs_symbol,bs_reportedCurrency,bs_cik,bs_fillingDate,bs_acceptedDate,bs_calendarYear,bs_period,...,is_link,is_finalLink,gross_profit,gross_ROA,EBIT_to_MP,rank_gross_ROA,rank_EBIT_to_MP,combined_score,final_rank,ranking_year
101,GIS,25.0,1999-05-30,GIS,USD,40704.0,1999-08-23,1999-08-23 00:00:00,1999.0,FY,...,https://www.sec.gov/Archives/edgar/data/40704/...,https://www.sec.gov/Archives/edgar/data/40704/...,3.846800e+09,3.489161,7.380633,15.0,3.0,18.0,1.0,2000
56,CPB,25.0,1999-08-01,CPB,USD,16732.0,1999-10-12,1999-10-12 00:00:00,1999.0,FY,...,https://www.sec.gov/Archives/edgar/data/16732/...,https://www.sec.gov/Archives/edgar/data/16732/...,3.571000e+09,2.759660,6.668085,25.0,5.0,30.0,2.0,2000
132,K,24.0,1999-12-31,K,USD,55067.0,2000-03-24,2000-03-24 00:00:00,1999.0,FY,...,https://www.sec.gov/Archives/edgar/data/55067/...,https://www.sec.gov/Archives/edgar/data/55067/...,3.947100e+09,2.515358,1.674127,33.0,9.0,42.0,3.0,2000
89,EXC,25.0,1999-12-31,EXC,USD,1109357.0,1999-12-31,1999-12-30 19:00:00,1999.0,FY,...,NaN,NaN,3.291600e+09,2.714498,0.927611,27.0,16.0,43.0,4.0,2000
45,CL,25.0,1999-12-31,CL,USD,21665.0,2000-03-27,2000-03-27 00:00:00,1999.0,FY,...,https://www.sec.gov/Archives/edgar/data/21665/...,https://www.sec.gov/Archives/edgar/data/21665/...,5.234400e+09,2.222864,1.079784,38.0,13.0,51.0,5.0,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9875,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,https://www.sec.gov/Archives/edgar/data/107263...,https://www.sec.gov/Archives/edgar/data/107263...,8.428000e+09,NaN,NaN,NaN,NaN,NaN,NaN,2025
9877,WST,0.0,2024-12-31,WST,USD,105770.0,2025-02-18,2025-02-18 16:44:56,2024.0,FY,...,https://www.sec.gov/Archives/edgar/data/105770...,https://www.sec.gov/Archives/edgar/data/105770...,9.985000e+08,0.706103,NaN,220.0,NaN,NaN,NaN,2025
9878,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,https://www.sec.gov/Archives/edgar/data/114053...,https://www.sec.gov/Archives/edgar/data/114053...,9.930000e+09,NaN,NaN,NaN,NaN,NaN,NaN,2025
9882,YUM,0.0,2024-12-31,YUM,USD,1041061.0,2025-02-19,2025-02-19 17:26:39,2024.0,FY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025


In [11]:
ranked_df

,bs_Symbol,bs_Unnamed: 1,bs_date,bs_symbol,bs_reportedCurrency,bs_cik,bs_fillingDate,bs_acceptedDate,bs_calendarYear,bs_period,...,is_link,is_finalLink,gross_profit,gross_ROA,EBIT_to_MP,rank_gross_ROA,rank_EBIT_to_MP,combined_score,final_rank,ranking_year
101,GIS,25.0,1999-05-30,GIS,USD,40704.0,1999-08-23,1999-08-23 00:00:00,1999.0,FY,...,https://www.sec.gov/Archives/edgar/data/40704/...,https://www.sec.gov/Archives/edgar/data/40704/...,3.846800e+09,3.489161,7.380633,15.0,3.0,18.0,1.0,2000
56,CPB,25.0,1999-08-01,CPB,USD,16732.0,1999-10-12,1999-10-12 00:00:00,1999.0,FY,...,https://www.sec.gov/Archives/edgar/data/16732/...,https://www.sec.gov/Archives/edgar/data/16732/...,3.571000e+09,2.759660,6.668085,25.0,5.0,30.0,2.0,2000
132,K,24.0,1999-12-31,K,USD,55067.0,2000-03-24,2000-03-24 00:00:00,1999.0,FY,...,https://www.sec.gov/Archives/edgar/data/55067/...,https://www.sec.gov/Archives/edgar/data/55067/...,3.947100e+09,2.515358,1.674127,33.0,9.0,42.0,3.0,2000
89,EXC,25.0,1999-12-31,EXC,USD,1109357.0,1999-12-31,1999-12-30 19:00:00,1999.0,FY,...,NaN,NaN,3.291600e+09,2.714498,0.927611,27.0,16.0,43.0,4.0,2000
45,CL,25.0,1999-12-31,CL,USD,21665.0,2000-03-27,2000-03-27 00:00:00,1999.0,FY,...,https://www.sec.gov/Archives/edgar/data/21665/...,https://www.sec.gov/Archives/edgar/data/21665/...,5.234400e+09,2.222864,1.079784,38.0,13.0,51.0,5.0,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9875,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,https://www.sec.gov/Archives/edgar/data/107263...,https://www.sec.gov/Archives/edgar/data/107263...,8.428000e+09,NaN,NaN,NaN,NaN,NaN,NaN,2025
9877,WST,0.0,2024-12-31,WST,USD,105770.0,2025-02-18,2025-02-18 16:44:56,2024.0,FY,...,https://www.sec.gov/Archives/edgar/data/105770...,https://www.sec.gov/Archives/edgar/data/105770...,9.985000e+08,0.706103,NaN,220.0,NaN,NaN,NaN,2025
9878,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,https://www.sec.gov/Archives/edgar/data/114053...,https://www.sec.gov/Archives/edgar/data/114053...,9.930000e+09,NaN,NaN,NaN,NaN,NaN,NaN,2025
9882,YUM,0.0,2024-12-31,YUM,USD,1041061.0,2025-02-19,2025-02-19 17:26:39,2024.0,FY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025


In [9]:
import yfinance as yf
import pandas as pd

portfolio_returns = []

for year in range(2000, 2025):
    buy_date = f"{year}-04-01"
    sell_date = f"{year+1}-03-30"
    
    top20 = ranked_df[ranked_df['ranking_year'] == year].nsmallest(20, 'final_rank')
    tickers = top20['symbol'].dropna().unique().tolist()

    if len(tickers) == 0:
        continue

    # Download adjusted close prices
    price_data = yf.download(tickers, start=buy_date, end=sell_date)['Close']

    # Ensure price_data is a DataFrame
    if isinstance(price_data, pd.Series):
        price_data = price_data.to_frame()

    # Forward-fill to handle missing weekends/holidays
    price_data = price_data.ffill().bfill()

    # Convert buy/sell dates to Timestamps and find closest available dates
    buy_date_actual = price_data.index[price_data.index.get_indexer([pd.Timestamp(buy_date)], method='nearest')[0]]
    sell_date_actual = price_data.index[price_data.index.get_indexer([pd.Timestamp(sell_date)], method='nearest')[0]]

    # Extract prices on closest available dates
    buy_prices = price_data.loc[buy_date_actual]
    sell_prices = price_data.loc[sell_date_actual]

    # Calculate returns
    returns = (sell_prices / buy_prices) - 1
    avg_return = returns.mean()

    portfolio_returns.append({
        'year': year,
        'buy_date': str(buy_date_actual.date()),
        'sell_date': str(sell_date_actual.date()),
        'average_return': avg_return,
        'top20_tickers': tickers
    })

# Create final results DataFrame
results_df = pd.DataFrame(portfolio_returns)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  20 of 20 completed

1 Failed download:
['HCA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-04-01 -> 2001-03-30) (Yahoo error = "Data doesn\'t exist for startDate = 954565200, endDate = 985928400")')
[*********************100%***********************]  20 of 20 completed

2 Failed downloads:
['HCA', 'PLL']: YFPricesMissingError('possibly delisted; no price data found  (1d 2001-04-01 -> 2002-03-30) (Yahoo error = "Data doesn\'t exist for startDate = 986101200, endDate = 1017464400")')
[*********************100%***********************]  20 of 20 completed

1 Failed download:
['HCA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2002-04-01 -> 2003-03-30) (Yahoo error = "Data doesn\'t exist for startDate = 1017637200, endDate = 1049000400")')
[*********************100%***********************]  20 of 20 completed

1 Failed download:
['HCA']: YFPricesMissingError('possibly delisted; no price da

In [10]:
results_df

,year,buy_date,sell_date,average_return,top20_tickers
0,2000,2000-04-03,2001-03-29,0.374879,"[GIS, CPB, K, EXC, CL, PEP, HCA, KR, SLM, DLX,..."
1,2001,2001-04-02,2002-03-28,0.233820,"[PLL, DLX, CPB, EHC, COP, EOG, K, KR, CL, NYT,..."
2,2002,2002-04-01,2003-03-28,-0.072632,"[YUM, DLX, GIS, HCA, EOG, SLM, K, CL, KR, MO, ..."
3,2003,2003-04-01,2004-03-29,0.421360,"[YUM, DLX, CL, EFX, KR, K, CNP, TROW, HCA, PPL..."
4,2004,2004-04-01,2005-03-29,0.185571,"[YUM, HCA, CPB, CL, EFX, SPG, K, EOG, KR, VZ, ..."
5,2005,2005-04-01,2006-03-29,0.178355,"[PLL, YUM, HCA, SPG, FNMA, CL, EFX, CPB, PGR, ..."
6,2006,2006-04-03,2007-03-29,0.204547,"[HCA, CL, YUM, FNMA, R, EFX, KR, VZ, CPB, K, A..."
7,2007,2007-04-02,2008-03-28,0.001782,"[YUM, R, CL, KR, DRI, EFX, UPS, K, VZ, XOM, MC..."
8,2008,2008-04-01,2009-03-27,-0.296433,"[R, CLX, CL, EL, KR, RSG, CPB, DRI, VZ, EXC, S..."
9,2009,2009-04-01,2010-03-29,0.527616,"[R, UPS, CL, XOM, K, KR, DRI, CPB, SPG, PEP, E..."
